# Fine-tune the TraceLayer Gemma agent

Run this notebook in Google Colab with a GPU runtime. First accept the Gemma 4 model terms with the Kaggle account tied to your Kaggle API credentials. Add `KAGGLE_USERNAME` and `KAGGLE_KEY` to Colab Secrets and enable notebook access to those secrets. This trains a small LoRA adapter; it does not train the base model from scratch.

The data is synthetic and contains no customer transactions. The notebook evaluates the untuned model, trains on the project examples, evaluates the adapter, then packages only the adapter and metrics.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_DIR = Path('/content/Code_Hunters')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Akullganesh19/Code_Hunters.git', str(REPO_DIR)], check=True)
PROJECT = REPO_DIR
GEMMA_DIR = PROJECT / 'upgrades' / 'gemma-agent'
print('Project:', PROJECT)

In [ ]:
# Read credentials from Colab Secrets without displaying or saving them.
from google.colab import userdata

for key in ('KAGGLE_USERNAME', 'KAGGLE_KEY'):
    value = userdata.get(key)
    if not value:
        raise RuntimeError(f'Missing Colab Secret: {key}. Add it under Secrets and enable notebook access.')
    os.environ[key] = value
os.environ['KERAS_BACKEND'] = 'jax'
print('Kaggle credentials loaded from Colab Secrets; values are hidden.')

In [ ]:
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
if gpu.returncode != 0:
    raise RuntimeError('No NVIDIA GPU detected. In Colab, choose Runtime > Change runtime type > GPU, then rerun.')
print(gpu.stdout.strip())
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', '-r', str(GEMMA_DIR / 'requirements.txt')], check=True)
print('Restart the runtime if Colab asks, then rerun the notebook from the top.')

In [ ]:
PREPARED = Path('/content/tracelayer-prepared')
OUTPUT = Path('/content/tracelayer-gemma-lora')
PRESET = 'gemma4_instruct_2b'
SEQUENCE_LENGTH = 1024
DTYPE = 'bfloat16'

subprocess.run([sys.executable, str(GEMMA_DIR / 'src' / 'prepare_data.py'), '--output-dir', str(PREPARED)], cwd=PROJECT, check=True)
print('Prepared data:', PREPARED)

In [ ]:
# Measure the untuned base model on the held-out examples first. This downloads the gated base weights.
subprocess.run([sys.executable, str(GEMMA_DIR / 'src' / 'evaluate.py'), '--prepared-eval', str(PREPARED / 'eval.jsonl'), '--preset', PRESET, '--sequence-length', str(SEQUENCE_LENGTH), '--dtype', DTYPE], cwd=GEMMA_DIR, check=True)

In [ ]:
subprocess.run([sys.executable, str(GEMMA_DIR / 'src' / 'train.py'), '--train', str(PREPARED / 'train.jsonl'), '--eval', str(PREPARED / 'eval.jsonl'), '--output', str(OUTPUT), '--preset', PRESET, '--sequence-length', str(SEQUENCE_LENGTH), '--dtype', DTYPE], cwd=GEMMA_DIR, check=True)

In [ ]:
subprocess.run([sys.executable, str(GEMMA_DIR / 'src' / 'evaluate.py'), '--prepared-eval', str(PREPARED / 'eval.jsonl'), '--adapter', str(OUTPUT / 'adapter.weights.h5'), '--preset', PRESET, '--sequence-length', str(SEQUENCE_LENGTH), '--dtype', DTYPE], cwd=GEMMA_DIR, check=True)
archive = subprocess.run(['zip', '-j', '-r', '/content/tracelayer-gemma-lora.zip', str(OUTPUT / 'adapter.weights.h5'), str(OUTPUT / 'training.json')], check=True, capture_output=True, text=True)
print('Adapter package: /content/tracelayer-gemma-lora.zip')
from google.colab import files
files.download('/content/tracelayer-gemma-lora.zip')

## Connect the adapter locally

Unzip the download into a private local folder, then follow `upgrades/gemma-agent/README.md` to start `serve.py` and the isolated agent API. Keep the adapter and Kaggle credentials out of Git. If the model fails to fit in GPU memory, use a larger GPU runtime; changing the base preset requires checking that the Gemma 4 prompt format remains compatible.